# Assignment 2 Research Paper

Steps
- Investigate TorNet data and pull out data only centered over large urban areas. Ensure we have enough coverage of tornado vs non tornado data
- Once subset of data is created, pull in PWS data from Netamo API for these locations
- Adds PWS data to model (somehow)

In [6]:
!pip install xarray
!pip install netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 13.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [netCDF4]m1/2 [netCDF4]


In [7]:
import sys

# Uncomment if tornet isn't installed in your environment or in your path already
sys.path.append('tornet')  

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# try:
#     from tornet.data.loader import read_file, TornadoDataLoader
# except ImportError as e:
#     print("WARNING: cannot import tornet library. Many cells below won't work. Install tornet or run sys.path.append('../') and rerun this cell")
#     raise e

# Set path to tornet dataset
# Assumes .tar.gz files are extracted.
# this directory should contain catalog.csv, train/ , test/
data_root= '/Users/evanshabsove/Documents/tornado_reserch_paper/tornet/tornet_data'

# Load catalog of all TORNET samples
catalog = pd.read_csv(os.path.join(data_root,'catalog.csv'),parse_dates=['start_time','end_time'])

# Setset catalog to get training data from certain years with strong tornaodes
years = [2013]
min_ef = 2

subset = catalog[
    (catalog.start_time.dt.year.isin(years)) & \
    (catalog['type']=='test') & \
    (catalog['ef_number']>=min_ef)
]
file_list = [os.path.join(data_root,f) for f in subset.filename]
print('Found',len(file_list),'files')

print("File exists:", os.path.exists(file_list[0]))
print("File path:", file_list[0])

# import xarray as xr

import xarray as xr
# print("Available engines:", list(xr.backends.list_engines()))
ds = xr.open_dataset(file_list[0], engine='netcdf4')
ds

Found 46 files
File exists: True
File path: /Users/evanshabsove/Documents/tornado_reserch_paper/tornet/tornet_data/test/2013/TOR_131004_030848_KOAX_472673_A1.nc


<xarray.Dataset> Size: 6MB
Dimensions:            (sweep: 2, time: 4, lims: 2, azimuth: 120, range: 240)
Coordinates:
  * time               (time) datetime64[ns] 32B 2013-10-04T02:53:30 ... 2013...
  * azimuth            (azimuth) float32 480B 164.2 164.8 165.2 ... 223.2 223.8
  * range              (range) float32 960B 4.235e+04 4.26e+04 ... 1.021e+05
Dimensions without coordinates: sweep, lims
Data variables:
    elevation          (sweep) float64 16B ...
    frame_labels       (time) uint8 4B ...
    nyquist_velocity   (time, sweep) float32 32B ...
    azimuth_limits     (lims) float32 8B ...
    range_limits       (lims) float32 8B ...
    DBZ                (time, azimuth, range, sweep) float32 922kB ...
    VEL                (time, azimuth, range, sweep) float32 922kB ...
    KDP                (time, azimuth, range, sweep) float32 922kB ...
    RHOHV              (time, azimuth, range, sweep) float32 922kB ...
    ZDR                (time, azimuth, range, sweep) float32 922kB ...
    WIDTH              (time, azimuth, range, sweep) float32 922kB ...
    range_folded_mask  (time, azimuth, range, sweep) uint8 230kB ...
Attributes:
    site_name:           KOAX
    site_lat:            41.320278
    site_lon:            -96.366667
    MissingDataFlag:     -999.0
    ef_number:           2.0
    event_id:            472673
    episode_id:          78589
    tornado_start_time:  2013-10-04 03:08:00
    tornado_end_time:    2013-10-04 03:34:00
    category:            TOR
    scit_id:             A1
    storm_event_url:     https://www.ncdc.noaa.gov/stormevents/eventdetails.j...

In [10]:
import pandas as pd
import math
import requests

def set_url(latll, lonll, latur, lonur, time):
    return f"https://madis-data.ncep.noaa.gov/madisPublic1/cgi-bin/madisXmlPublicDir?rdr=&time={time}&minbck=-59&minfwd=0&recwin=3&dfltrsel=1&state=&latll={latll}&lonll={lonll}&latur={latur}&lonur={lonur}&stanam=&stasel=0&pvdrsel=1&varsel=2&qctype=0&qcsel=1&xml=1&csvmiss=0&pvd=APRSWXNET"

def get_bounding_box(lat, lon, distance_km=20):
    """
    Returns the bounding box corners (latll, lonll, latur, lonur) for a box centered at (lat, lon)
    with the given distance_km as the box width/height.
    latll: latitude of lower left (SW) corner
    lonll: longitude of lower left (SW) corner
    latur: latitude of upper right (NE) corner
    lonur: longitude of upper right (NE) corner
    """
    # 1 degree latitude is ~111 km
    delta_lat = (distance_km / 2) / 111.0

    # 1 degree longitude is ~111 km * cos(latitude)
    delta_lon = (distance_km / 2) / (111.0 * math.cos(math.radians(lat)))

    latll = lat - delta_lat
    lonll = lon - delta_lon
    latur = lat + delta_lat
    lonur = lon + delta_lon

    return latll, lonll, latur, lonur

def convert_to_timestamp(time):
    """
    Converts a datetime object or numpy.datetime64 to a string in the format 'YYYYMMDD_HHMM'.
    """
    import numpy as np
    import datetime
    if isinstance(time, np.datetime64):
        # Convert numpy.datetime64 to python datetime
        time = pd.to_datetime(str(time)).to_pydatetime()
    elif hasattr(time, 'to_pydatetime'):
        time = time.to_pydatetime()
    return time.strftime('%Y%m%d_%H%M')

def download_madis_data(latll, lonll, latur, lonur, time, filename="madis_data.xml"):
    """
    Downloads MADIS data for the given bounding box and time.
    Checks if data exists and prints "YES HAS DATA" if found.
    Returns the URL used for the request.
    """
    import os
    import xml.etree.ElementTree as ET
    
    if os.path.exists(filename):
        print(f"File already exists: {filename}, skipping download.")
        return filename
    
    timestamp = convert_to_timestamp(time)
    url = set_url(latll, lonll, latur, lonur, timestamp)
    print("Downloading data from:", url)
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        # Parse XML to check if there's actual data
        root = ET.fromstring(response.text)
        records = root.findall('.//record')
        
        if len(records) > 0:
            print(f"✅ YES HAS DATA - Found {len(records)} records")
        else:
            print("❌ No data records found")
        
        # Optionally save the file
        # with open(filename, "w") as f:
        #     f.write(response.text)
        # print(f"Data saved to {filename}")
        
    except Exception as e:
        print(f"Error downloading data: {e}")
    
    return url

def get_id_from_storm_event_url(storm_event_url):
    """
    Extracts the 'id' parameter value from the storm_event_url.
    Returns the id as a string, or None if not found.
    """
    import urllib.parse
    if storm_event_url:
        parsed = urllib.parse.urlparse(storm_event_url)
        query = urllib.parse.parse_qs(parsed.query)
        return query.get('id', [None])[0]
    return None

# Loop through tornet files from 2013 and grab the maxiumum SW corner latitude - south, SW corner longitude - west, NE corner latitude - north, NE corner longitude - east
# Setset catalog to get training data from certain years with strong tornaodes
years = [2013]

subset = catalog[
    (catalog.start_time.dt.year.isin(years))
]
file_list = [os.path.join(data_root,f) for f in subset.filename]
print('Found',len(file_list),'files')

print("File exists:", os.path.exists(file_list[0]))
print("File path:", file_list[0])

for fpath in file_list:
    # print(f"Processing file: {fpath}")
    ds = xr.open_dataset(fpath, engine='netcdf4')
    latll, lonll, latur, lonur = get_bounding_box(ds.attrs['site_lat'], ds.attrs['site_lon'])
    storm_id = get_id_from_storm_event_url(ds.attrs['storm_event_url'])
    print(f"Storm ID: {storm_id}")
    # print(f"Time : {ds['time'].values[0]}")
    download_madis_data(latll, lonll, latur, lonur, ds['time'].values[0], filename=f"tornet/tornet_data/madis_data/madis_data_{storm_id}_{ds['time'].values[0]}.xml")

Found 4071 files
File exists: True
File path: /Users/evanshabsove/Documents/tornado_reserch_paper/tornet/tornet_data/train/2013/NUL_130831_234007_KBGM_467173s_B5.nc
Storm ID: 467173
✅ YES HAS DATA - Found 7 records
Storm ID: 467173


KeyboardInterrupt: 

In [12]:
# Download using the reconstructed URL (no state parameter)
if 'url_no_state' in locals():
    print("Testing API request WITHOUT state parameter...")
    print(f"URL: {url_no_state}\n")
    
    try:
        response = requests.get(url_no_state)
        response.raise_for_status()
        
        # Parse the response
        root = ET.fromstring(response.text)
        records = root.findall('.//record')
        
        if len(records) > 0:
            print(f"✅ YES HAS DATA - Found {len(records)} records")
            print("\nSample records:")
            for i, record in enumerate(records[:5]):  # Show first 5
                var = record.get('var')
                value = record.get('data_value')
                lat = record.get('lat')
                lon = record.get('lon')
                print(f"  {i+1}. {var}: {value} at ({lat}, {lon})")
        else:
            print("❌ No data records found")
            print(f"Response content: {response.text[:500]}")
            
    except Exception as e:
        print(f"❌ Error downloading data: {e}")
else:
    print("Please run the previous cell first to generate the URL")

Testing API request WITHOUT state parameter...
URL: https://madis-data.ncep.noaa.gov/madisPublic1/cgi-bin/madisXmlPublicDir?rdr=&time=20130901_2028&minbck=-59&minfwd=0&recwin=3&dfltrsel=1&state=&latll=35.57546590990991&lonll=-78.60061109358337&latur=35.755646090090096&lonur=-78.37883290641663&stanam=&stasel=0&pvdrsel=1&varsel=2&qctype=0&qcsel=1&xml=1&csvmiss=0&pvd=APRSWXNET



KeyboardInterrupt: 

### Test API Request Without State

Now let's actually test if the MADIS API works without the `state` parameter by making a real request:

In [ ]:
import os
import xml.etree.ElementTree as ET

# Directory containing MADIS XML files
madis_dir = "madis_data"

# Set to collect unique 'var' values
unique_vars = set()

# Loop through all XML files in the directory
for fname in os.listdir(madis_dir):
    if fname.endswith(".xml"):
        fpath = os.path.join(madis_dir, fname)
        try:
            tree = ET.parse(fpath)
            root = tree.getroot()
            for record in root.findall(".//record"):
                var = record.attrib.get("var")
                if var:
                    unique_vars.add(var)
        except Exception as e:
            print(f"Error parsing {fname}: {e}")

# Convert to sorted list for easier viewing
unique_vars_list = sorted(unique_vars)
print("Unique 'var' values found in MADIS XML files:")
for v in unique_vars_list:
    print(v)


FileNotFoundError: [Errno 2] No such file or directory: 'madis_data'

In [ ]:
import xml.etree.ElementTree as ET

def extract_madis_features(xml_file):
    """
    Extracts relevant MADIS features from the XML file.
    Returns a dictionary of features for all known variables.
    """
    features = {}
    var_map = {
        'V-ALTSE': 'madis_atmospheric_pressure',
        'V-DD': 'madis_wind_direction',
        'V-FF': 'madis_wind_speed',
        'V-FFGUST': 'madis_wind_gust_speed',
        'V-RH': 'madis_relative_humidity',
        'V-T': 'madis_temperature',
        'V-TD': 'madis_temperature_dew_point',
    }
    try:
        import xml.etree.ElementTree as ET
        tree = ET.parse(xml_file)
        root = tree.getroot()
        for record in root.findall('.//record'):
            var = record.attrib.get('var')
            if var in var_map:
                try:
                    features[var_map[var]] = float(record.attrib.get('data_value'))
                except (TypeError, ValueError):
                    features[var_map[var]] = None
    except Exception as e:
        print(f"Error parsing {xml_file}: {e}")
    return features

# Loop through the first 100 files and add MADIS data as attributes to each xarray
for fpath in file_list[0:100]:
    print(f"Processing file: {fpath}")
    ds = xr.open_dataset(fpath, engine='netcdf4')
    storm_id = get_id_from_storm_event_url(ds.attrs['storm_event_url'])
    madis_xml_file = f"madis_data/madis_data_{storm_id}_{ds['time'].values[0]}.xml"
    madis_features = extract_madis_features(madis_xml_file)
    
    # Add MADIS features as attributes to the xarray dataset
    ds = ds.assign_attrs(**madis_features)
    print(f"Added MADIS features to xarray: {madis_features}")

Processing file: tornet/tornet_data/train/2013/NUL_130831_234007_KBGM_467173s_B5.nc
Added MADIS features to xarray: {'madis_temperature_dew_point': 292.35556, 'madis_relative_humidity': 92.0, 'madis_temperature': 293.705566, 'madis_wind_direction': 135.0, 'madis_wind_speed': 0.0, 'madis_wind_gust_speed': 0.0, 'madis_atmospheric_pressure': 101010.0}
Processing file: tornet/tornet_data/train/2013/NUL_130831_234422_KBGM_467173s_K7.nc
Added MADIS features to xarray: {'madis_temperature_dew_point': 292.35556, 'madis_relative_humidity': 92.0, 'madis_temperature': 293.705566, 'madis_wind_direction': 135.0, 'madis_wind_speed': 0.0, 'madis_wind_gust_speed': 0.0, 'madis_atmospheric_pressure': 101010.0}
Processing file: tornet/tornet_data/train/2013/NUL_130831_234837_KBGM_467173s_X3.nc
Added MADIS features to xarray: {'madis_temperature_dew_point': 292.35556, 'madis_relative_humidity': 92.0, 'madis_temperature': 293.705566, 'madis_wind_direction': 135.0, 'madis_wind_speed': 0.0, 'madis_wind_gust_

In [ ]:
# Create CNN model with MADIS data included

# Step 1: Build the model without MADIS data


ModuleNotFoundError: No module named 'tornet.data'

In [ ]:
# Step 2: Add MADIS data to the model

